### Preamble

In [1]:
from app.network.api import load_sumo_network
from app.network.road import build_lane_records, build_junction_records, compute_bounds, compute_edge_markings, compute_lane_markings
from app.network.opposite_marking import compute_opposite_direction_markings
from app.network.util import SVG

from pathlib import Path

output_dir = "opposite-marking"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

## Visualizing the opposite marking detection algorithm

### Cropping the network to a smaller area

In [2]:
root, road_file = load_sumo_network('tue-small')

# Construct lane records
lane_records = build_lane_records(root)
junction_records = build_junction_records(root)

# Collect the road polygons
lane_polys = [rec["polygon"] for rec in lane_records]
junc_polys = [rec["polygon"] for rec in junction_records]

# Compute bounds for viewport fitting
bounds = compute_bounds(lane_polys + junc_polys)

# Compute lane markings
lane_markings = compute_lane_markings(lane_records)
edge_markings = compute_edge_markings(lane_polys + junc_polys)
opposite_markings, opposite_marking_debug = compute_opposite_direction_markings(lane_records)

For a small part of the network, we are going to illustrate the opposite lane marking algorithm. Therefore, we first filter the lane and junction records to a small area.

In [3]:
from shapely.geometry import box

def crop_to_bounds(polygon_records, bounds):
    filtered_records = []
    for rec in polygon_records:
        if rec["polygon"].intersects(bounds):
            cropped_rec = dict(rec)  # Create a copy of the record
            cropped_rec["polygon"] = rec["polygon"].intersection(bounds)
            filtered_records.append(cropped_rec)
    
    return filtered_records

subset_bounds = { "minx": 425, "miny": -540, "maxx": 465, "maxy": -510 }
bounds_box = box(subset_bounds["minx"], subset_bounds["miny"], subset_bounds["maxx"], subset_bounds["maxy"])

# Draw cropping box on the full network
svg = SVG(bounds)
svg.draw_polygons(lane_polys + junc_polys)
svg.draw_polygons([bounds_box], stroke="green", stroke_width=0.5)
svg.write(output_dir / "tue-small.svg")

lane_records_cropped = crop_to_bounds(lane_records, bounds_box)
junction_records_cropped = crop_to_bounds(junction_records, bounds_box)

lane_polys_cropped = [rec["polygon"] for rec in lane_records_cropped]
junc_polys_cropped = [rec["polygon"] for rec in junction_records_cropped]

# Draw the cropped network with cropping box
svg = SVG(subset_bounds)
svg.draw_polygons(lane_polys_cropped + junc_polys_cropped)
svg.draw_polygons([bounds_box], stroke="green", stroke_width=0.2)
svg.write(output_dir / "tue-small-cropped.svg")

### Step 1: Finding candidates

In [4]:
def base():
    svg = SVG(subset_bounds)
    svg.draw_polygons(lane_polys_cropped, stroke="grey", fill="grey", stroke_width=0.2)
    svg.draw_polygons(junc_polys_cropped, stroke="grey", fill="grey", stroke_width=0.2)
    return svg

svg = base()
svg.write(output_dir / "tue-cropped-markings-1.svg")

In [5]:
# Compute lane markings
lane_markings = compute_lane_markings(lane_records_cropped)
edge_markings = compute_edge_markings(lane_polys_cropped + junc_polys_cropped)
opposite_markings, opposite_marking_debug = compute_opposite_direction_markings(lane_records_cropped)

In [14]:
ego = 1
candidates = opposite_marking_debug["band_a"][ego]["candidates"]

# highlight all candidates
svg.draw_polygons([lane_records_cropped[j]["polygon"] for j in candidates], stroke="red", fill="red", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-2.svg")

# highlight one candidate
candidate = 9
svg = base()
svg.draw_polygons([lane_records_cropped[candidate]["polygon"]], stroke="purple", fill="purple", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-3.svg")

### Step 2: Buffer and compute intersection

In [ ]:
svg = base()
band_1 = opposite_marking_debug["band_a"][ego]["polygon"]
band_2 = opposite_marking_debug["band_a"][candidate2]["polygon"]
overlap = opposite_marking_debug["overlap"][0]["polygon"]
svg.draw_polygons([band_1], fill="red", stroke_width=0)
svg.draw_polygons([band_2], fill="red", stroke_width=0)
svg.draw_polygons([overlap], fill="blue", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-2.svg")